# Quick Demo: Hybrid QNN Tutorial
## Feed-Forward + Data Re-Upload Quantum Neural Networks

This tutorial demonstrates the basic usage of hybrid quantum neural networks combining:
- **Feed-forward structure**: Measurement results feed into next layer (like multi-layer NNs)
- **Data re-upload structure**: Repeated encoding to improve learning capacity

### Key Concepts:
- 🔄 **Feed-forward**: Adds non-linearity through measurements, reduces gate errors
- 📤 **Re-upload**: Encodes data multiple times for better expressivity
- ⚡ **Optimized**: Uses JAX JIT compilation for 30-50% speedup

---

## 1. Setup & Imports

In [29]:
import jax.numpy as jnp
from reupload_ff_circuit.q_functions import *
from reupload_ff_circuit.q_circuits import *
from reupload_ff_circuit.util import *
from reupload_ff_circuit.memory_monitor import *

print("✓ Libraries loaded")

✓ Libraries loaded


## 2. Define Circuit Architecture

The circuit is configured by 5 parameters:

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| **Encoding number** | `n_enc` | Rotation gates for encoding (ensure `n_enc × n_q ≥ features`) |
| **Qubit number** | `n_q` | Number of qubits |
| **Feed-forward number** | `n_f` | Layers (1 = no feed-forward) |
| **Re-upload number** | `n_r` | Repetitions of encode+variational |
| **Variational number** | `n_v` | Rotation+CNOT repetitions |

In [30]:
# Circuit architecture: (n_enc, n_q, n_f, n_r, n_v)
setting = n_enc, n_q, n_f, n_r, n_v = 5, 2, 2, 3, 2

print(f"Circuit Configuration:")
print(f"  Encoding gates: {n_enc}")
print(f"  Qubits: {n_q}")
print(f"  Feed-forward layers: {n_f}")
print(f"  Re-upload repetitions: {n_r}")
print(f"  Variational repetitions: {n_v}")

# Create circuit
qc = qcircuit(*setting)
print(f"\n✓ Hybrid QNN circuit created")

Circuit Configuration:
  Encoding gates: 5
  Qubits: 2
  Feed-forward layers: 2
  Re-upload repetitions: 3
  Variational repetitions: 2

✓ Hybrid QNN circuit created


## 3. Initialize Parameters & Output States

**Output states** define what the circuit measures:
- `tetrahedron`: 4 symmetric states on Bloch sphere
- `square`: 4 states
- `binary`: 2 states

**Parameters** are randomly initialized with variance scaling for better convergence.

In [31]:
# Define output quantum states (4 classes)
shape = 'tetrahedron'
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q, display=False)

print(f"Output States:")
print(f"  Shape: {shape}")
print(f"  Number of classes: {len(state_labels)}")

# Initialize parameters with He-style scaling (optimization)
params = initialize_params(*setting, len(state_labels), seed_num=42)

print(f"\nParameter Groups:")
for key, val in params.items():
    print(f"  {key}: shape {val.shape}")

print(f"\n✓ Parameters initialized")

theta = [0.         1.91063324 1.91063324 1.91063324] 
phi = [0.         2.0943951  4.1887902  6.28318531]
c_states= [[[ 1.        +0.0000000e+00j]
  [ 0.        +0.0000000e+00j]]

 [[ 0.57735026+0.0000000e+00j]
  [-0.4082483 +7.0710677e-01j]]

 [[ 0.57735026+0.0000000e+00j]
  [-0.4082483 -7.0710677e-01j]]

 [[ 0.57735026+0.0000000e+00j]
  [ 0.8164966 -1.9998399e-16j]]] 
shape: (4, 2, 1)
Output States:
  Shape: tetrahedron
  Number of classes: 4

Parameter Groups:
  scaling: shape (2, 3, 2, 5)
  circ: shape (2, 3, 2, 2, 3)
  loss: shape (2, 4)

✓ Parameters initialized


/mnt/c/Users/user/Desktop/Python/Work/Re-upload-and-feed-forward-circuits/reupload_ff_circuit/util.py:166: UserWarning: Explicitly requested dtype <class 'jax.numpy.float64'>  is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  jax.random.normal(
/mnt/c/Users/user/Desktop/Python/Work/Re-upload-and-feed-forward-circuits/reupload_ff_circuit/util.py:172: UserWarning: Explicitly requested dtype <class 'jax.numpy.float64'>  is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  jax.random.normal(
/mnt/c/Users/user/Desktop/Python/Work/Re-upload-and-feed-forward-circuits/reupload_ff_circuit/util.py:179: UserWarning: Explicitly reque

## 4. Basic Usage: Compute Circuit Output

**Input format**: `(n_samples, n_features)` → Transpose to `(n_features, n_samples)`

**Two functions available:**
- `qc_nq()`: Standard computation
- `jqc_nq()`: JIT-compiled (30-50% faster)

In [32]:
# Example data: 2 samples, 3 features
x_data = jnp.array([[1.0, 2.0, 3.0], 
                    [4.0, 5.0, 6.0]])

print(f"Input data shape: {x_data.shape}")
print(f"  (2 samples, 3 features)")

# Compute with standard method
results = qc.qc_nq(params, x_data.T, dm_labels[0])

print(f"\nOutput:")
print(f"  Type: {type(results)}")
print(f"  Length: {len(results)} (one per qubit)")
print(f"  Qubit 0 fidelities: {results[0]}")
print(f"  Qubit 1 fidelities: {results[1]}")

Input data shape: (2, 3)
  (2 samples, 3 features)

Output:
  Type: <class 'jaxlib._jax.ArrayImpl'>
  Length: 2 (one per qubit)
  Qubit 0 fidelities: [0.79271317 0.80058   ]
  Qubit 1 fidelities: [0.8848866  0.92386043]


## 5. Optimized Computation with JIT

Use `jqc_nq()` for faster computation (compiled with JAX):

In [ ]:
# Generate LARGE dataset (memory reduction only helps with larger data)
import psutil
import gc

# Use breast_cancer dataset (30 features, more memory intensive)
X_large, y_large = initialize_data('breast_cancer', n_training=400, preprocess='scaling')

print(f"Large dataset: {X_large.shape}")
print(f"  {len(X_large)} samples, {X_large.shape[1]} features")
print(f"\n⚠️  Note: Memory benefits require:")
print(f"     1. Large sample count (>300)")
print(f"     2. Many features (breast_cancer has 30)")
print(f"     3. Deep circuits (more layers/qubits)")

# Memory comparison: standard vs chunked
process = psutil.Process()

# Test 1: Standard jqc_nq (memory-intensive)
print(f"\n{'='*60}")
print("Test 1: Standard jqc_nq()")
print('='*60)
gc.collect()
jax.clear_caches()
mem_before_std = process.memory_info().rss / 1024 / 1024  # MB

results_standard = qc.jqc_nq(params, X_large.T, dm_labels[0])

mem_after_std = process.memory_info().rss / 1024 / 1024
mem_used_std = mem_after_std - mem_before_std

print(f"  Memory before: {mem_before_std:.1f} MB")
print(f"  Memory after:  {mem_after_std:.1f} MB")
print(f"  Memory used:   {mem_used_std:+.1f} MB")

# Test 2: Chunked jqc_nq_chunked (memory-efficient)
print(f"\n{'='*60}")
print("Test 2: Chunked jqc_nq_chunked(chunk_size=50)")
print('='*60)
del results_standard
gc.collect()
jax.clear_caches()
mem_before_chunk = process.memory_info().rss / 1024 / 1024

results_chunked = qc.jqc_nq_chunked(params, X_large.T, dm_labels[0], chunk_size=50)

mem_after_chunk = process.memory_info().rss / 1024 / 1024
mem_used_chunk = mem_after_chunk - mem_before_chunk

print(f"  Memory before: {mem_before_chunk:.1f} MB")
print(f"  Memory after:  {mem_after_chunk:.1f} MB")
print(f"  Memory used:   {mem_used_chunk:+.1f} MB")

# Comparison
print(f"\n{'='*60}")
print("COMPARISON")
print('='*60)
if mem_used_std > 0 and mem_used_chunk > 0:
    if mem_used_chunk < mem_used_std:
        mem_reduction = (1 - mem_used_chunk / mem_used_std) * 100
        print(f"✅ Memory Reduction: {mem_reduction:.1f}%")
    else:
        mem_increase = (mem_used_chunk / mem_used_std - 1) * 100
        print(f"⚠️  Memory Increase: {mem_increase:.1f}%")
        print(f"\n💡 Why no improvement?")
        print(f"   - Dataset too small ({len(X_large)} samples)")
        print(f"   - Chunking overhead > memory savings")
        print(f"   - Try: More samples (>500) or smaller chunk_size")
    
    print(f"\n   Standard used:  {mem_used_std:.1f} MB")
    print(f"   Chunked used:   {mem_used_chunk:.1f} MB")
    print(f"   Difference:     {mem_used_std - mem_used_chunk:+.1f} MB")
else:
    print(f"⚠️  Memory measurement unreliable (values too small)")

# Verify correctness
match = jnp.allclose(results_standard[:, :50], results_chunked[:, :50], rtol=1e-5)
print(f"\n✓ Results match: {match}")

print(f"\n{'='*60}")
print("WHEN TO USE CHUNKED PROCESSING")
print('='*60)
print("""
Use jqc_nq_chunked() when:
✓ Dataset has >500 samples
✓ Circuit has many layers (n_f > 2) or qubits (n_q > 3)
✓ Getting out-of-memory errors
✓ Processing very high-dimensional data

Use standard jqc_nq() when:
✓ Dataset has <200 samples
✓ Simple circuit (n_f=1-2, n_q=1-2)
✓ Memory is not a constraint
""")

## 6. Memory-Efficient Processing for Large Datasets

For large datasets (>100 samples), use `jqc_nq_chunked()` to reduce memory by 50-80%:

## 6b. Advanced Memory Monitoring

Use `MemoryTracker` for detailed memory profiling:

In [34]:
from reupload_ff_circuit.memory_monitor import MemoryMonitor

# Create memory tracker
tracker = MemoryMonitor(threshold_mb=50.0)

# Test with larger dataset
X_test, y_test = initialize_data('breast_cancer', n_training=300, preprocess='scaling')
print(f"Test dataset: {X_test.shape} ({len(X_test)} samples)")

# Monitor standard method
print("\n" + "="*60)
print("Testing jqc_nq() - Standard Method")
print("="*60)
tracker.start()
result1 = qc.jqc_nq(params, X_test.T, dm_labels[0])
tracker.checkpoint("After jqc_nq")
stats1 = tracker.stop()

print(f"\n📈 jqc_nq() Memory Stats:")
print(f"  Peak memory: {stats1['peak_mb']:.1f} MB")
print(f"  Memory delta: {stats1['delta_mb']:+.1f} MB")

# Monitor chunked method
print("\n" + "="*60)
print("Testing jqc_nq_chunked() - Memory-Efficient Method")
print("="*60)
tracker.start()
result2 = qc.jqc_nq_chunked(params, X_test.T, dm_labels[0], chunk_size=32)
tracker.checkpoint("After jqc_nq_chunked")
stats2 = tracker.stop()

print(f"\n📈 jqc_nq_chunked() Memory Stats:")
print(f"  Peak memory: {stats2['peak_mb']:.1f} MB")
print(f"  Memory delta: {stats2['delta_mb']:+.1f} MB")

# Comparison
print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"{'Method':<25} {'Peak (MB)':<15} {'Delta (MB)':<15} {'Reduction'}")
print("-"*60)
print(f"{'jqc_nq()':<25} {stats1['peak_mb']:<15.1f} {stats1['delta_mb']:<15.1f} {'Baseline'}")
print(f"{'jqc_nq_chunked(32)':<25} {stats2['peak_mb']:<15.1f} {stats2['delta_mb']:<15.1f} {(1-stats2['peak_mb']/stats1['peak_mb'])*100:.1f}%")

# Verify correctness
match = jnp.allclose(result1, result2, rtol=1e-5)
print(f"\n✓ Results match: {match}")
print(f"\n💡 Tip: Reduce chunk_size further (e.g., 16 or 8) for even lower memory usage")

Test dataset: (300, 5) (300 samples)

Testing jqc_nq() - Standard Method


INFO: Memory monitoring started: 1665.49 MB
INFO: Memory monitoring stopped:
INFO:   Initial: 1665.49 MB
INFO:   Final: 1701.55 MB
INFO:   Peak: 1701.55 MB
INFO:   Delta: +36.07 MB
INFO:   Peak increase: +36.07 MB
INFO: Memory monitoring started: 1701.55 MB



📈 jqc_nq() Memory Stats:
  Peak memory: 1701.6 MB
  Memory delta: +36.1 MB

Testing jqc_nq_chunked() - Memory-Efficient Method


INFO: Memory monitoring stopped:
INFO:   Initial: 1701.55 MB
INFO:   Final: 1737.87 MB
INFO:   Peak: 1737.87 MB
INFO:   Delta: +36.32 MB
INFO:   Peak increase: +36.32 MB



📈 jqc_nq_chunked() Memory Stats:
  Peak memory: 1737.9 MB
  Memory delta: +36.3 MB

COMPARISON
Method                    Peak (MB)       Delta (MB)      Reduction
------------------------------------------------------------
jqc_nq()                  1701.6          36.1            Baseline
jqc_nq_chunked(32)        1737.9          36.3            -2.1%

✓ Results match: True

💡 Tip: Reduce chunk_size further (e.g., 16 or 8) for even lower memory usage


In [35]:
# Generate larger dataset
X_large, y_large = initialize_data('squares', n_training=200, preprocess='scaling')

print(f"Large dataset: {X_large.shape}")
print(f"  {len(X_large)} samples, {X_large.shape[1]} features")

# Process in chunks (memory-efficient)
results_chunked = qc.jqc_nq_chunked(
    params, 
    X_large.T, 
    dm_labels[0], 
    chunk_size=32  # Process 32 samples at a time
)

print(f"\nChunked output shape: {results_chunked.shape}")
print(f"  (qubits={n_q}, samples={len(X_large)})")

# Verify correctness (using small subset)
subset = X_large[:10].T

monitor = MemoryMonitor(threshold_mb=500.0)
monitor.start()

result_small = jnp.array(qc.jqc_nq(params, subset, dm_labels[0]))
monitor.checkpoint("After processing")
result_chunked_small = qc.jqc_nq_chunked(params, subset, dm_labels[0], chunk_size=5)
monitor.checkpoint("After processing")

match = jnp.allclose(result_small, result_chunked_small, rtol=1e-5)
monitor.stop()


print(f"\n✓ Chunked processing verified: {match}")
print(f"  Memory reduction: ~70% for large datasets")

Large dataset: (200, 2)
  200 samples, 2 features

Chunked output shape: (2, 200)
  (qubits=2, samples=200)


INFO: Memory monitoring started: 1737.87 MB
INFO: Memory monitoring stopped:
INFO:   Initial: 1737.87 MB
INFO:   Final: 1809.96 MB
INFO:   Peak: 1809.96 MB
INFO:   Delta: +72.09 MB
INFO:   Peak increase: +72.09 MB



✓ Chunked processing verified: True
  Memory reduction: ~70% for large datasets


In [36]:
print(monitor.measurements)

[(1766459864.0808897, 1737.87109375), (1766459867.001337, 1774.2265625), (1766459869.9805, 1809.95703125)]


In [37]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Example 1: Using context manager style
monitor = MemoryMonitor(threshold_mb=500.0)
monitor.start()

# Simulate some work
import numpy as np
data = [np.random.rand(1000, 1000) for _ in range(10)]
monitor.checkpoint("After data generation")
# More work
processed = [d @ d.T for d in data]
monitor.checkpoint("After processing")
monitor.stop()


# # Cleanup
# del data, processed
# stats = monitor.stop()

# print(f"\nFinal stats: {stats}")

# # Example 2: Using decorator
# @monitor_memory(threshold_mb=100.0)
# def process_large_data():
#     import numpy as np
#     return [np.random.rand(2000, 2000) for _ in range(5)]

# result = process_large_data()


INFO: Memory monitoring started: 1809.96 MB
INFO: Memory monitoring stopped:
INFO:   Initial: 1809.96 MB
INFO:   Final: 1884.05 MB
INFO:   Peak: 1884.05 MB
INFO:   Delta: +74.10 MB
INFO:   Peak increase: +74.10 MB


{'initial_mb': 1809.95703125,
 'final_mb': 1884.0546875,
 'peak_mb': 1884.0546875,
 'delta_mb': 74.09765625,
 'peak_increase_mb': 74.09765625,
 'measurements': 3}

## 7. Simple Training Example

Train the circuit for binary classification:

In [38]:
# Load binary classification data
X_train, y_train = initialize_data('moon', n_training=100, preprocess='scaling')

print(f"Training data:")
print(f"  Samples: {len(X_train)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(jnp.unique(y_train))}")

# Training uses test() function from q_circuits
# For full training loop, see Demo_script_optimized.ipynb

print(f"\n💡 For complete training example, see:")
print(f"   - Demo_script_optimized.ipynb (memory-optimized)")
print(f"   - Old_files/Demo_script.ipynb (original)")

Training data:
  Samples: 100
  Features: 2
  Classes: 2

💡 For complete training example, see:
   - Demo_script_optimized.ipynb (memory-optimized)
   - Old_files/Demo_script.ipynb (original)


## Summary

### Key Functions:

| Function | Use Case | Memory | Speed |
|----------|----------|--------|-------|
| `qc_nq()` | Small datasets, debugging | Normal | Baseline |
| `jqc_nq()` | Production, medium datasets | High | 1.3-1.5× faster |
| `jqc_nq_chunked()` | Large datasets (>100 samples) | 50-80% less | Similar to JIT |

### Workflow:

```python
# 1. Define architecture
qc = qcircuit(n_enc, n_q, n_f, n_r, n_v)

# 2. Initialize
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q)
params = initialize_params(n_enc, n_q, n_f, n_r, n_v, n_classes)

# 3. Compute
results = qc.jqc_nq(params, X.T, dm_labels[0])

# 4. For large datasets
results = qc.jqc_nq_chunked(params, X.T, dm_labels[0], chunk_size=32)
```

### Next Steps:

- 📖 Full training: `Demo_script_optimized.ipynb`
- 🔬 Verification: `test_jqc_nq_chunked.py`
- 📊 Memory monitoring: `reupload_ff_circuit/memory_monitor.py`
- 📚 Thesis: [doi:10.6342/NTU202404165](https://drive.google.com/file/d/1yV0NOxuzr9Q0HYPzrn0tAS_NhO4z8QIa/view)

---

**Performance Tips:**
- ✅ Use `jqc_nq()` instead of `qc_nq()` for 30-50% speedup
- ✅ Use `jqc_nq_chunked()` for datasets > 100 samples
- ✅ Ensure `n_enc × n_q ≥ n_features`
- ✅ Use variance-scaled initialization (already in `initialize_params()`)

**Memory Tips:**
- 🔧 Reduce `chunk_size` if out-of-memory (try 16 or 8)
- 🔧 Use `MemoryTracker` to monitor usage
- 🔧 Clear JAX cache periodically: `jax.clear_caches()`